In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

In [2]:
file_path = "../data/data_Maastricht_2025.xlsx"

xls = pd.ExcelFile(file_path)

print(xls.sheet_names)

['Service Point Locations', 'Daily Activity', 'CBS Squares', 'Nodes', 'Edges']


In [3]:
# Load sheets
daily_activity = pd.read_excel(file_path, sheet_name="Daily Activity")
service_points = pd.read_excel(file_path, sheet_name="Service Point Locations")
cbs_squares = pd.read_excel(file_path, sheet_name="CBS Squares")
nodes = pd.read_excel(file_path, sheet_name="Nodes")
edges = pd.read_excel(file_path, sheet_name="Edges")

In [4]:
print(daily_activity.columns)
print(service_points.columns)
print(cbs_squares.columns)
print(nodes.columns)
print(edges.columns)
daily_activity.head()
service_points.head()
cbs_squares.head()
nodes.head()
edges.head()

Index(['Date', 'Day Index', 'Day of Week', 'Location ID', 'Deliveries',
       'Pickups'],
      dtype='object')
Index(['Location ID', 'X', 'Y', 'Containing Square'], dtype='object')
Index(['Square', 'Population', 'Male', 'Female', 'Age0-14', 'Age15-24',
       'Age25-44', 'Age45-64', 'Age65+', 'Households',
       'Single-person households', 'Multi-person households w/o kids',
       'Single parent households', 'Two-parent households', 'Houses',
       'Home ownership %', 'Rental %', 'Social housing %', 'Vacant houses',
       'Avg. home value k€', 'OAD', 'Urbanization index',
       'Median household income', 'Percentage low income households',
       'Percentage high income households',
       'Distance nearest supermarket in km', 'X', 'Y'],
      dtype='object')
Index(['NODE ID', 'X', 'Y', 'SQUARE'], dtype='object')
Index(['EDGE ID', 'V1', 'V2', 'DIST', 'ONE_WAY', 'TYPE', 'NAME', 'MAX_SPEED',
       'Unnamed: 8', 'X1', 'Y1', 'X2', 'Y2', 'Square 1', 'Square 2',
       'Square mid'],

,EDGE ID,V1,V2,DIST,ONE_WAY,TYPE,NAME,MAX_SPEED,Unnamed: 8,X1,Y1,X2,Y2,Square 1,Square 2,Square mid
0,e10525,7866,7867,50.068154,True,busway,Koningin Emmaplein,5,NaN,43649,47502,43228,47231,E1755N3175,E1750N3175,E1755N3175
1,e10940,4982,4985,121.148050,True,residential,Statensingel,30,NaN,46296,49361,47289,50055,E1755N3175,E1755N3180,E1755N3180
2,e10939,8170,4982,19.334167,True,residential,Statensingel,30,NaN,46137,49251,46296,49361,E1755N3175,E1755N3175,E1755N3175
3,e10950,8181,8182,6.935416,False,service,undefined,30,NaN,75025,19083,75072,19134,E1775N3145,E1775N3145,E1775N3145
4,e11163,8320,3405,24.475498,True,residential,Pergamijndonk,30,NaN,33850,61288,33939,61060,E1745N3190,E1745N3190,E1745N3190


In [5]:
# Create graph
G = nx.DiGraph()

# Add edges
for _, row in edges.iterrows():

    start = row["V1"]
    end = row["V2"]
    dist = row["DIST"]

    # add forward edge
    G.add_edge(start, end, weight=dist)

    # if road is not one-way, add reverse edge
    if row["ONE_WAY"] == False:
        G.add_edge(end, start, weight=dist)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 8098
Edges: 19009


In [6]:
# Create dictionary of node coordinates

node_positions = {}

for _, row in nodes.iterrows():

    node_id = row["NODE ID"]
    x = row["X"]
    y = row["Y"]

    node_positions[node_id] = (x, y)

In [7]:
def find_nearest_node(x, y, node_positions):

    closest_node = None
    min_distance = float("inf")

    for node_id, (nx_pos, ny_pos) in node_positions.items():

        dist = ((x - nx_pos)**2 + (y - ny_pos)**2)**0.5

        if dist < min_distance:

            min_distance = dist
            closest_node = node_id

    return closest_node

In [8]:
service_points["nearest_node"] = service_points.apply(
    lambda row: find_nearest_node(row["X"], row["Y"], node_positions),
    axis=1
)

service_points.head()

,Location ID,X,Y,Containing Square,nearest_node
0,753,75319,28665,E1775N3155,753
1,3343,36589,60425,E1750N3190,3343
2,1934,48959,42794,E1755N3170,1934
3,5820,98158,61912,E1790N3190,5820
4,4356,13591,30750,E1730N3155,4356


In [9]:
cbs_squares["nearest_node"] = cbs_squares.apply(
    lambda row: find_nearest_node(row["X"], row["Y"], node_positions),
    axis=1
)

cbs_squares.head()

,Square,Population,Male,Female,Age0-14,Age15-24,Age25-44,Age45-64,Age65+,Households,...,Avg. home value k€,OAD,Urbanization index,Median household income,Percentage low income households,Percentage high income households,Distance nearest supermarket in km,X,Y,nearest_node
0,E1725N3160,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3676.249942,33402.280,6159
1,E1725N3170,175.0,95.0,80.0,15.0,30.0,15.0,70.0,45.0,70.0,...,409.0,880.0,4.0,60-80 above middle,NaN,NaN,1.5,3676.249942,42640.610,1974
2,E1725N3180,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3676.249942,51878.940,122
3,E1725N3185,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3676.249942,56498.105,4794
4,E1725N3190,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3676.249942,61117.270,114


In [10]:
# Keep only squares with population data

cbs_clean = cbs_squares.dropna(subset=["Population"]).copy()

print("Original squares:", len(cbs_squares))
print("Clean squares:", len(cbs_clean))

cbs_clean[[
    "Square",
    "Population",
    "Households",
    "Age25-44",
    "nearest_node"
]].head()

Original squares: 284
Clean squares: 216


,Square,Population,Households,Age25-44,nearest_node
1,E1725N3170,175.0,70.0,15.0,1974
8,E1730N3160,435.0,220.0,75.0,5940
9,E1730N3165,1815.0,950.0,375.0,1398
10,E1730N3170,1975.0,805.0,400.0,1965
11,E1730N3175,140.0,50.0,20.0,2282


In [11]:
# Create assignment list

assignments = []
for _, square in cbs_clean.iterrows():
    square_node = square["nearest_node"]
    best_sp = None
    best_distance = float("inf")
    for _, sp in service_points.iterrows():
        sp_node = sp["nearest_node"]
        try:
            dist = nx.shortest_path_length(
                G,
                source=square_node,
                target=sp_node,
                weight="weight"
            )
            if dist < best_distance:
                best_distance = dist
                best_sp = sp["Location ID"]
        except:
            continue
    assignments.append({
        "Square": square["Square"],
        "Population": square["Population"],
        "Assigned_SP": best_sp,
        "Distance": best_distance
    })

assignments_df = pd.DataFrame(assignments)
assignments_df.head()    

,Square,Population,Assigned_SP,Distance
0,E1725N3170,175.0,2075,892.392379
1,E1730N3160,435.0,1379,612.981853
2,E1730N3165,1815.0,1379,631.173038
3,E1730N3170,1975.0,2075,518.720476
4,E1730N3175,140.0,2075,689.310773


In [12]:
# Convert distance to km
assignments_df["Distance_km"] = assignments_df["Distance"] / 1000

# Summary statistics
print("\nDistance Summary (km)")
print(assignments_df["Distance_km"].describe())

# Coverage metrics
within_1km = (assignments_df["Distance_km"] <= 1).mean() * 100
within_2km = (assignments_df["Distance_km"] <= 2).mean() * 100
over_3km = (assignments_df["Distance_km"] > 3).mean() * 100

print(f"\nSquares within 1 km: {within_1km:.1f}%")
print(f"Squares within 2 km: {within_2km:.1f}%")
print(f"Squares over 3 km: {over_3km:.1f}%")

# Top 10 most remote CBS squares
print("\nMost remote CBS squares:")
display(
    assignments_df.sort_values("Distance_km", ascending=False)
    .head(10)
)


Distance Summary (km)
count    216.000000
mean       1.436600
std        1.236621
min        0.000000
25%        0.614685
50%        0.933321
75%        1.845843
max        5.943292
Name: Distance_km, dtype: float64

Squares within 1 km: 51.9%
Squares within 2 km: 76.4%
Squares over 3 km: 12.0%

Most remote CBS squares:


,Square,Population,Assigned_SP,Distance,Distance_km
215,E1820N3215,55.0,5820,5943.291883,5.943292
213,E1815N3215,90.0,5820,5385.659697,5.385660
212,E1815N3210,45.0,5820,5347.348828,5.347349
201,E1810N3215,815.0,5820,4727.482945,4.727483
174,E1795N3215,120.0,5820,4677.552962,4.677553
202,E1810N3220,265.0,5820,4632.555930,4.632556
190,E1805N3220,945.0,5820,4593.462579,4.593463
160,E1790N3220,160.0,5820,4507.194811,4.507195
200,E1810N3210,80.0,5820,4366.556164,4.366556
179,E1800N3210,110.0,5820,4338.866204,4.338866


In [13]:
def pickup_probability(distance):

    if distance <= 1:
        return 0.80

    elif distance <= 2:
        return 0.65

    elif distance <= 3:
        return 0.50

    elif distance <= 4:
        return 0.35

    else:
        return 0.20

In [15]:
# =====================================================
# STEP 6 - ESTIMATE DEMAND PER CBS SQUARE
# =====================================================

TOTAL_PARCELS = 1351852

# Remove duplicated population columns if they exist
for col in ["Population_x", "Population_y"]:
    if col in assignments_df.columns:
        assignments_df = assignments_df.drop(columns=[col])

# If Population is already in assignments_df, use it directly
# Otherwise merge it from cbs_clean
if "Population" not in assignments_df.columns:
    assignments_df = assignments_df.merge(
        cbs_clean[["Square", "Population"]],
        on="Square",
        how="left"
    )

# Total population represented by valid CBS squares
total_population = assignments_df["Population"].sum()

# Parcels generated per resident per year
demand_factor = TOTAL_PARCELS / total_population

# Estimate annual parcel demand per square
assignments_df["Annual_Demand"] = (
    assignments_df["Population"] * demand_factor
)

# Results
print(f"Total population: {total_population:,.0f}")
print(f"Demand factor: {demand_factor:.4f} parcels per resident per year")
print(f"Estimated parcels: {assignments_df['Annual_Demand'].sum():,.0f}")

# Preview
display(
    assignments_df[
        ["Square",
         "Population",
         "Assigned_SP",
         "Distance_km",
         "Annual_Demand"]
    ].head()
)

Total population: 134,580
Demand factor: 10.0450 parcels per resident per year
Estimated parcels: 1,351,852


,Square,Population,Assigned_SP,Distance_km,Annual_Demand
0,E1725N3170,175.0,2075,0.892392,1757.869669
1,E1730N3160,435.0,1379,0.612982,4369.561748
2,E1730N3165,1815.0,1379,0.631173,18231.619706
3,E1730N3170,1975.0,2075,0.518720,19838.814831
4,E1730N3175,140.0,2075,0.689311,1406.295735


In [16]:
# =====================================================
# STEP 7 - PICKUP PROBABILITY AND DEMAND SPLIT
# =====================================================

def pickup_probability(distance_km):

    if distance_km <= 1:
        return 0.80

    elif distance_km <= 2:
        return 0.65

    elif distance_km <= 3:
        return 0.50

    elif distance_km <= 4:
        return 0.35

    else:
        return 0.20

# Apply probability function
assignments_df["Pickup_Probability"] = (
    assignments_df["Distance_km"]
    .apply(pickup_probability)
)

# Split annual demand
assignments_df["Annual_Pickups"] = (
    assignments_df["Annual_Demand"]
    * assignments_df["Pickup_Probability"]
)

assignments_df["Annual_Deliveries"] = (
    assignments_df["Annual_Demand"]
    - assignments_df["Annual_Pickups"]
)

# Summary
total_pickups = assignments_df["Annual_Pickups"].sum()
total_deliveries = assignments_df["Annual_Deliveries"].sum()

pickup_share = total_pickups / TOTAL_PARCELS * 100
delivery_share = total_deliveries / TOTAL_PARCELS * 100

print(f"Total pickups: {total_pickups:,.0f}")
print(f"Total deliveries: {total_deliveries:,.0f}")

print(f"Pickup share: {pickup_share:.1f}%")
print(f"Delivery share: {delivery_share:.1f}%")

# Preview
display(
    assignments_df[
        [
            "Square",
            "Assigned_SP",
            "Distance_km",
            "Annual_Demand",
            "Pickup_Probability",
            "Annual_Pickups",
            "Annual_Deliveries"
        ]
    ].head()
)

Total pickups: 1,002,008
Total deliveries: 349,844
Pickup share: 74.1%
Delivery share: 25.9%


,Square,Assigned_SP,Distance_km,Annual_Demand,Pickup_Probability,Annual_Pickups,Annual_Deliveries
0,E1725N3170,2075,0.892392,1757.869669,0.8,1406.295735,351.573934
1,E1730N3160,1379,0.612982,4369.561748,0.8,3495.649398,873.912350
2,E1730N3165,1379,0.631173,18231.619706,0.8,14585.295765,3646.323941
3,E1730N3170,2075,0.518720,19838.814831,0.8,15871.051865,3967.762966
4,E1730N3175,2075,0.689311,1406.295735,0.8,1125.036588,281.259147


In [17]:
# =====================================================
# STEP 8 - COST CALCULATION
# =====================================================

FIXED_COST = 50000
DELIVERY_COST_PER_KM = 1.5
STORAGE_COST_PER_PACKAGE = 0.1

# Service points currently used
active_sps = assignments_df["Assigned_SP"].nunique()

# Fixed cost
fixed_cost = active_sps * FIXED_COST

# Delivery cost
assignments_df["Delivery_Cost"] = (
    assignments_df["Annual_Deliveries"]
    * assignments_df["Distance_km"]
    * DELIVERY_COST_PER_KM
)

# Storage cost
assignments_df["Storage_Cost"] = (
    assignments_df["Annual_Pickups"]
    * STORAGE_COST_PER_PACKAGE
)

total_delivery_cost = assignments_df["Delivery_Cost"].sum()
total_storage_cost = assignments_df["Storage_Cost"].sum()

total_cost = (
    fixed_cost
    + total_delivery_cost
    + total_storage_cost
)

print("===== COST BREAKDOWN =====")
print(f"Active service points: {active_sps}")
print(f"Fixed cost: €{fixed_cost:,.0f}")
print(f"Delivery cost: €{total_delivery_cost:,.0f}")
print(f"Storage cost: €{total_storage_cost:,.0f}")
print(f"TOTAL COST: €{total_cost:,.0f}")

# Cost by service point
sp_summary = (
    assignments_df
    .groupby("Assigned_SP")
    .agg(
        Population=("Population", "sum"),
        Demand=("Annual_Demand", "sum"),
        Pickups=("Annual_Pickups", "sum"),
        Deliveries=("Annual_Deliveries", "sum"),
        Avg_Distance=("Distance_km", "mean"),
        Delivery_Cost=("Delivery_Cost", "sum"),
        Storage_Cost=("Storage_Cost", "sum")
    )
    .reset_index()
)

display(
    sp_summary.sort_values(
        "Delivery_Cost",
        ascending=False
    ).head(10)
)

===== COST BREAKDOWN =====
Active service points: 35
Fixed cost: €1,750,000
Delivery cost: €693,354
Storage cost: €100,201
TOTAL COST: €2,543,555


,Assigned_SP,Population,Demand,Pickups,Deliveries,Avg_Distance,Delivery_Cost,Storage_Cost
28,5820,14285.0,143492.389805,70849.681372,72642.708434,3.094782,370558.728557,7084.968137
31,7229,11385.0,114361.978154,82381.306398,31980.671757,1.410978,49925.508596,8238.130640
4,753,5810.0,58361.272997,42033.175019,16328.097979,1.147654,25698.406135,4203.317502
33,7859,4880.0,49019.451330,36081.530569,12937.920761,1.812262,24309.745103,3608.153057
2,657,3740.0,37568.186060,27930.037792,9638.148269,1.674279,20837.225696,2793.003779
11,1739,7975.0,80108.632040,62045.265574,18063.366466,0.718455,18744.301362,6204.526557
34,8282,6670.0,66999.946797,52477.432092,14522.514705,0.728879,15685.123228,5247.743209
12,1934,3085.0,30988.731015,21950.769676,9037.961339,0.674419,11780.727185,2195.076968
22,4387,4330.0,43494.718086,33025.348588,10469.369498,1.268367,11480.793044,3302.534859
16,3155,2755.0,27673.891069,20421.423064,7252.468004,1.172110,9853.490457,2042.142306


In [18]:
# =====================================================
# BASELINE SOLUTION
# =====================================================

baseline_cost = total_cost

print("BASELINE SOLUTION")
print("-" * 40)
print(f"Total cost: €{baseline_cost:,.0f}")
print(f"Service points used: {active_sps}")

print("\nTop 10 most expensive service points:")

display(
    sp_summary[
        [
            "Assigned_SP",
            "Demand",
            "Avg_Distance",
            "Delivery_Cost"
        ]
    ]
    .sort_values(
        "Delivery_Cost",
        ascending=False
    )
    .head(10)
)

BASELINE SOLUTION
----------------------------------------
Total cost: €2,543,555
Service points used: 35

Top 10 most expensive service points:


,Assigned_SP,Demand,Avg_Distance,Delivery_Cost
28,5820,143492.389805,3.094782,370558.728557
31,7229,114361.978154,1.410978,49925.508596
4,753,58361.272997,1.147654,25698.406135
33,7859,49019.451330,1.812262,24309.745103
2,657,37568.186060,1.674279,20837.225696
11,1739,80108.632040,0.718455,18744.301362
34,8282,66999.946797,0.728879,15685.123228
12,1934,30988.731015,0.674419,11780.727185
22,4387,43494.718086,1.268367,11480.793044
16,3155,27673.891069,1.172110,9853.490457


In [22]:
# =====================================================
# STEP 9 - LOCAL SEARCH / REASSIGNMENT HEURISTIC
# =====================================================

import random
import copy

# -----------------------------
# Parameters
# -----------------------------

DELIVERY_COST_PER_KM = 1.5
STORAGE_COST_PER_PACKAGE = 0.1

MAX_DISTANCE_KM = 6.0          # avoid assigning a square to very far SPs
N_NEAREST_CANDIDATES = 8       # only test nearest alternative service points
MAX_ITERATIONS = 20000
NO_IMPROVEMENT_LIMIT = 3000

random.seed(42)

# -----------------------------
# Helper: pickup probability
# -----------------------------

def pickup_probability(distance_km):
    if distance_km <= 1:
        return 0.80
    elif distance_km <= 2:
        return 0.65
    elif distance_km <= 3:
        return 0.50
    elif distance_km <= 4:
        return 0.35
    else:
        return 0.20


# -----------------------------
# Helper: cost of one CBS square
# -----------------------------

def square_cost(annual_demand, distance_km):
    
    p_pickup = pickup_probability(distance_km)
    
    pickups = annual_demand * p_pickup
    deliveries = annual_demand - pickups
    
    delivery_cost = deliveries * distance_km * DELIVERY_COST_PER_KM
    storage_cost = pickups * STORAGE_COST_PER_PACKAGE
    
    total = delivery_cost + storage_cost
    
    return total, pickups, deliveries


# -----------------------------
# Precompute distances:
# each CBS square -> each service point
# -----------------------------

print("Precomputing distances from CBS squares to service points...")

distance_dict = {}

for _, square in cbs_clean.iterrows():
    
    square_id = square["Square"]
    square_node = square["nearest_node"]
    
    distance_dict[square_id] = {}
    
    for _, sp in service_points.iterrows():
        
        sp_id = sp["Location ID"]
        sp_node = sp["nearest_node"]
        
        try:
            dist_km = nx.shortest_path_length(
                G,
                source=square_node,
                target=sp_node,
                weight="weight"
            ) / 1000
            
            distance_dict[square_id][sp_id] = dist_km
            
        except:
            continue

print("Distance precomputation completed.")


# -----------------------------
# Build initial solution
# -----------------------------

solution = assignments_df.copy()

# Make sure required columns exist
solution["Current_SP"] = solution["Assigned_SP"]
solution["Current_Distance_km"] = solution["Distance_km"]

# Recalculate cost for consistency
cost_data = solution.apply(
    lambda row: square_cost(
        row["Annual_Demand"],
        row["Current_Distance_km"]
    ),
    axis=1
)

solution["Current_Cost"] = [x[0] for x in cost_data]
solution["Current_Pickups"] = [x[1] for x in cost_data]
solution["Current_Deliveries"] = [x[2] for x in cost_data]

baseline_variable_cost = solution["Current_Cost"].sum()
baseline_total_cost = fixed_cost + baseline_variable_cost

print("\nInitial solution")
print("-" * 40)
print(f"Fixed cost: €{fixed_cost:,.0f}")
print(f"Variable cost: €{baseline_variable_cost:,.0f}")
print(f"Total cost: €{baseline_total_cost:,.0f}")


# -----------------------------
# Candidate service points
# for each square
# -----------------------------

candidate_sps = {}

for square_id, distances in distance_dict.items():
    
    sorted_sps = sorted(
        distances.items(),
        key=lambda x: x[1]
    )
    
    # keep only reasonable nearby candidates
    filtered = [
        sp_id for sp_id, dist in sorted_sps
        if dist <= MAX_DISTANCE_KM
    ]
    
    candidate_sps[square_id] = filtered[:N_NEAREST_CANDIDATES]


# -----------------------------
# Local Search
# -----------------------------

best_solution = solution.copy()
current_solution = solution.copy()

best_cost = baseline_total_cost
current_cost = baseline_total_cost

accepted_moves = 0
iterations_without_improvement = 0

cost_history = [best_cost]

print("\nRunning local search...")

for iteration in range(MAX_ITERATIONS):
    
    if iterations_without_improvement >= NO_IMPROVEMENT_LIMIT:
        break
    
    # choose random square
    idx = random.choice(current_solution.index.tolist())
    
    square_id = current_solution.loc[idx, "Square"]
    current_sp = current_solution.loc[idx, "Current_SP"]
    annual_demand = current_solution.loc[idx, "Annual_Demand"]
    old_distance = current_solution.loc[idx, "Current_Distance_km"]
    old_cost = current_solution.loc[idx, "Current_Cost"]
    
    # possible alternative SPs
    possible_sps = [
        sp for sp in candidate_sps[square_id]
        if sp != current_sp
    ]
    
    if len(possible_sps) == 0:
        iterations_without_improvement += 1
        continue
    
    # test all candidate service points and pick best move
    best_move = None
    
    for new_sp in possible_sps:
        
        if new_sp not in distance_dict[square_id]:
            continue
        
        new_distance = distance_dict[square_id][new_sp]
        
        new_cost, new_pickups, new_deliveries = square_cost(
            annual_demand,
            new_distance
        )
        
        improvement = old_cost - new_cost
        
        if improvement > 0:
            
            if best_move is None or improvement > best_move["improvement"]:
                
                best_move = {
                    "new_sp": new_sp,
                    "new_distance": new_distance,
                    "new_cost": new_cost,
                    "new_pickups": new_pickups,
                    "new_deliveries": new_deliveries,
                    "improvement": improvement
                }
    
    # accept move if it improves
    if best_move is not None:
        
        current_solution.loc[idx, "Current_SP"] = best_move["new_sp"]
        current_solution.loc[idx, "Current_Distance_km"] = best_move["new_distance"]
        current_solution.loc[idx, "Current_Cost"] = best_move["new_cost"]
        current_solution.loc[idx, "Current_Pickups"] = best_move["new_pickups"]
        current_solution.loc[idx, "Current_Deliveries"] = best_move["new_deliveries"]
        
        current_cost = current_cost - best_move["improvement"]
        
        accepted_moves += 1
        iterations_without_improvement = 0
        
        if current_cost < best_cost:
            best_cost = current_cost
            best_solution = current_solution.copy()
            cost_history.append(best_cost)
    
    else:
        iterations_without_improvement += 1


# -----------------------------
# Results
# -----------------------------

improvement_abs = baseline_total_cost - best_cost
improvement_pct = improvement_abs / baseline_total_cost * 100

print("\nLOCAL SEARCH RESULTS")
print("=" * 50)
print(f"Iterations performed: {iteration + 1}")
print(f"Accepted moves: {accepted_moves}")
print(f"Baseline total cost: €{baseline_total_cost:,.0f}")
print(f"Improved total cost: €{best_cost:,.0f}")
print(f"Cost reduction: €{improvement_abs:,.0f}")
print(f"Cost reduction %: {improvement_pct:.2f}%")

print("\nPickup / delivery after local search")
print("-" * 50)

total_pickups_new = best_solution["Current_Pickups"].sum()
total_deliveries_new = best_solution["Current_Deliveries"].sum()

print(f"Pickups: {total_pickups_new:,.0f}")
print(f"Deliveries: {total_deliveries_new:,.0f}")
print(f"Pickup share: {total_pickups_new / TOTAL_PARCELS * 100:.1f}%")
print(f"Delivery share: {total_deliveries_new / TOTAL_PARCELS * 100:.1f}%")

print("\nDistance after local search")
print("-" * 50)
print(best_solution["Current_Distance_km"].describe())

# Summary by service point
optimized_sp_summary = (
    best_solution
    .groupby("Current_SP")
    .agg(
        Population=("Population", "sum"),
        Demand=("Annual_Demand", "sum"),
        Pickups=("Current_Pickups", "sum"),
        Deliveries=("Current_Deliveries", "sum"),
        Avg_Distance=("Current_Distance_km", "mean"),
        Variable_Cost=("Current_Cost", "sum"),
        Number_of_Squares=("Square", "count")
    )
    .reset_index()
    .sort_values("Variable_Cost", ascending=False)
)

display(optimized_sp_summary.head(10))

# Show changed assignments
changed_assignments = best_solution[
    best_solution["Assigned_SP"] != best_solution["Current_SP"]
].copy()

print(f"\nNumber of changed CBS square assignments: {len(changed_assignments)}")

display(
    changed_assignments[
        [
            "Square",
            "Population",
            "Assigned_SP",
            "Current_SP",
            "Distance_km",
            "Current_Distance_km",
            "Annual_Demand",
            "Current_Cost"
        ]
    ].head(20)
)

Precomputing distances from CBS squares to service points...
Distance precomputation completed.

Initial solution
----------------------------------------
Fixed cost: €1,750,000
Variable cost: €793,555
Total cost: €2,543,555

Running local search...

LOCAL SEARCH RESULTS
Iterations performed: 3001
Accepted moves: 0
Baseline total cost: €2,543,555
Improved total cost: €2,543,555
Cost reduction: €0
Cost reduction %: 0.00%

Pickup / delivery after local search
--------------------------------------------------
Pickups: 1,002,008
Deliveries: 349,844
Pickup share: 74.1%
Delivery share: 25.9%

Distance after local search
--------------------------------------------------
count    216.000000
mean       1.436600
std        1.236621
min        0.000000
25%        0.614685
50%        0.933321
75%        1.845843
max        5.943292
Name: Current_Distance_km, dtype: float64


,Current_SP,Population,Demand,Pickups,Deliveries,Avg_Distance,Variable_Cost,Number_of_Squares
28,5820,14285.0,143492.389805,70849.681372,72642.708434,3.094782,377643.696694,33
31,7229,11385.0,114361.978154,82381.306398,31980.671757,1.410978,58163.639236,9
4,753,5810.0,58361.272997,42033.175019,16328.097979,1.147654,29901.723636,9
33,7859,4880.0,49019.451330,36081.530569,12937.920761,1.812262,27917.898160,13
11,1739,7975.0,80108.632040,62045.265574,18063.366466,0.718455,24948.827920,4
2,657,3740.0,37568.186060,27930.037792,9638.148269,1.674279,23630.229475,15
34,8282,6670.0,66999.946797,52477.432092,14522.514705,0.728879,20932.866437,5
22,4387,4330.0,43494.718086,33025.348588,10469.369498,1.268367,14783.327903,8
12,1934,3085.0,30988.731015,21950.769676,9037.961339,0.674419,13975.804152,2
3,718,4940.0,49622.149502,39260.763427,10361.386075,0.737681,12969.787806,4



Number of changed CBS square assignments: 0


,Square,Population,Assigned_SP,Current_SP,Distance_km,Current_Distance_km,Annual_Demand,Current_Cost


In [23]:
# =====================================================
# STEP 10 - FULL LOCAL SEARCH HEURISTIC
# Operators: CLOSE + OPEN + SWAP/REASSIGN
# =====================================================

FIXED_COST = 50000
APL_FIXED_COST = 50000
DELIVERY_COST_PER_KM = 1.5
STORAGE_COST_PER_PACKAGE = 0.1

MIN_OPEN_LOCATIONS = 25
MAX_OPEN_LOCATIONS = 45

MAX_AVG_DISTANCE_KM = 2.30
MAX_DISTANCE_KM = 6.50

MAX_ITERATIONS = 30

# -----------------------------------------------------
# Pickup probability
# -----------------------------------------------------

def pickup_probability(distance_km):
    if distance_km <= 1:
        return 0.80
    elif distance_km <= 2:
        return 0.65
    elif distance_km <= 3:
        return 0.50
    elif distance_km <= 4:
        return 0.35
    else:
        return 0.20


# -----------------------------------------------------
# Build candidate APLs
# Candidate APLs are placed at high-population / remote CBS squares
# -----------------------------------------------------

apl_candidates = (
    assignments_df
    .copy()
    .sort_values(["Distance_km", "Population"], ascending=[False, False])
    .head(20)
)

apl_locations = {}

for i, row in apl_candidates.iterrows():
    apl_id = f"APL_{row['Square']}"
    apl_locations[apl_id] = {
        "Square": row["Square"],
        "nearest_node": cbs_clean.loc[
            cbs_clean["Square"] == row["Square"],
            "nearest_node"
        ].iloc[0],
        "fixed_cost": APL_FIXED_COST
    }

print(f"Created {len(apl_locations)} candidate APLs.")


# -----------------------------------------------------
# Existing service point locations
# -----------------------------------------------------

location_nodes = {}

for _, row in service_points.iterrows():
    location_nodes[row["Location ID"]] = {
        "nearest_node": row["nearest_node"],
        "fixed_cost": FIXED_COST,
        "type": "SP"
    }

for apl_id, apl_data in apl_locations.items():
    location_nodes[apl_id] = {
        "nearest_node": apl_data["nearest_node"],
        "fixed_cost": apl_data["fixed_cost"],
        "type": "APL"
    }


# -----------------------------------------------------
# Precompute distances from each square to SPs + APLs
# -----------------------------------------------------

print("Precomputing distances to SPs and APLs...")

full_distance_dict = {}

for _, square in cbs_clean.iterrows():

    square_id = square["Square"]
    square_node = square["nearest_node"]

    full_distance_dict[square_id] = {}

    for loc_id, loc_data in location_nodes.items():

        loc_node = loc_data["nearest_node"]

        try:
            dist_km = nx.shortest_path_length(
                G,
                source=square_node,
                target=loc_node,
                weight="weight"
            ) / 1000

            full_distance_dict[square_id][loc_id] = dist_km

        except:
            continue

print("Distance precomputation completed.")


# -----------------------------------------------------
# Evaluate network
# -----------------------------------------------------

def evaluate_network(open_locations):

    rows = []

    for _, square in cbs_clean.iterrows():

        square_id = square["Square"]
        population = square["Population"]
        annual_demand = population * demand_factor

        best_loc = None
        best_distance = float("inf")

        for loc_id in open_locations:

            if loc_id in full_distance_dict[square_id]:

                dist = full_distance_dict[square_id][loc_id]

                if dist < best_distance:
                    best_distance = dist
                    best_loc = loc_id

        pickup_prob = pickup_probability(best_distance)
        pickups = annual_demand * pickup_prob
        deliveries = annual_demand - pickups

        delivery_cost = deliveries * best_distance * DELIVERY_COST_PER_KM
        storage_cost = pickups * STORAGE_COST_PER_PACKAGE

        rows.append({
            "Square": square_id,
            "Population": population,
            "Assigned_Location": best_loc,
            "Distance_km": best_distance,
            "Annual_Demand": annual_demand,
            "Pickup_Probability": pickup_prob,
            "Pickups": pickups,
            "Deliveries": deliveries,
            "Variable_Cost": delivery_cost + storage_cost
        })

    solution = pd.DataFrame(rows)

    fixed_cost = sum(
        location_nodes[loc]["fixed_cost"]
        for loc in open_locations
    )

    variable_cost = solution["Variable_Cost"].sum()
    total_cost = fixed_cost + variable_cost

    avg_distance = solution["Distance_km"].mean()
    max_distance = solution["Distance_km"].max()

    feasible = (
        avg_distance <= MAX_AVG_DISTANCE_KM and
        max_distance <= MAX_DISTANCE_KM
    )

    return {
        "solution": solution,
        "open_locations": set(open_locations),
        "fixed_cost": fixed_cost,
        "variable_cost": variable_cost,
        "total_cost": total_cost,
        "avg_distance": avg_distance,
        "max_distance": max_distance,
        "feasible": feasible
    }


# -----------------------------------------------------
# Location efficiency summary
# -----------------------------------------------------

def location_summary(network_result):

    solution = network_result["solution"]

    summary = (
        solution
        .groupby("Assigned_Location")
        .agg(
            Population=("Population", "sum"),
            Demand=("Annual_Demand", "sum"),
            Pickups=("Pickups", "sum"),
            Deliveries=("Deliveries", "sum"),
            Avg_Distance=("Distance_km", "mean"),
            Max_Distance=("Distance_km", "max"),
            Variable_Cost=("Variable_Cost", "sum"),
            Squares=("Square", "count")
        )
        .reset_index()
    )

    summary["Fixed_Cost"] = summary["Assigned_Location"].apply(
        lambda loc: location_nodes[loc]["fixed_cost"]
    )

    summary["Type"] = summary["Assigned_Location"].apply(
        lambda loc: location_nodes[loc]["type"]
    )

    summary["Total_Location_Cost"] = (
        summary["Fixed_Cost"] + summary["Variable_Cost"]
    )

    summary["Cost_per_Parcel"] = (
        summary["Total_Location_Cost"] / summary["Demand"]
    )

    return summary.sort_values("Cost_per_Parcel", ascending=False)


# -----------------------------------------------------
# Initial network: all existing service points open
# -----------------------------------------------------

initial_open_locations = set(service_points["Location ID"].unique())

current = evaluate_network(initial_open_locations)
best = current

search_log = []

print("INITIAL NETWORK")
print("=" * 50)
print(f"Open locations: {len(current['open_locations'])}")
print(f"Total cost: €{current['total_cost']:,.0f}")
print(f"Fixed cost: €{current['fixed_cost']:,.0f}")
print(f"Variable cost: €{current['variable_cost']:,.0f}")
print(f"Average distance: {current['avg_distance']:.2f} km")
print(f"Maximum distance: {current['max_distance']:.2f} km")

print("\nLeast efficient baseline locations:")
display(location_summary(current).head(10))


# -----------------------------------------------------
# Local Search Loop
# -----------------------------------------------------

for iteration in range(1, MAX_ITERATIONS + 1):

    best_move = None

    current_open = current["open_locations"]

    # =================================================
    # OPERATOR 1: CLOSE
    # Test closing weak locations
    # =================================================

    if len(current_open) > MIN_OPEN_LOCATIONS:

        ranked_locations = location_summary(current)

        close_candidates = ranked_locations[
            ranked_locations["Type"] == "SP"
        ]["Assigned_Location"].head(10).tolist()

        for loc_to_close in close_candidates:

            test_open = current_open.copy()
            test_open.remove(loc_to_close)

            test_result = evaluate_network(test_open)

            improvement = current["total_cost"] - test_result["total_cost"]

            if test_result["feasible"] and improvement > 0:

                if best_move is None or improvement > best_move["improvement"]:
                    best_move = {
                        "operator": "CLOSE",
                        "location": loc_to_close,
                        "result": test_result,
                        "improvement": improvement
                    }

    # =================================================
    # OPERATOR 2: OPEN
    # Test opening APL candidates
    # =================================================

    if len(current_open) < MAX_OPEN_LOCATIONS:

        closed_apls = [
            apl_id for apl_id in apl_locations.keys()
            if apl_id not in current_open
        ]

        for apl_id in closed_apls:

            test_open = current_open.copy()
            test_open.add(apl_id)

            test_result = evaluate_network(test_open)

            improvement = current["total_cost"] - test_result["total_cost"]

            if test_result["feasible"] and improvement > 0:

                if best_move is None or improvement > best_move["improvement"]:
                    best_move = {
                        "operator": "OPEN",
                        "location": apl_id,
                        "result": test_result,
                        "improvement": improvement
                    }

    # =================================================
    # OPERATOR 3: CLOSE + OPEN SWAP
    # Close one inefficient SP and open one APL
    # =================================================

    ranked_locations = location_summary(current)

    close_candidates = ranked_locations[
        ranked_locations["Type"] == "SP"
    ]["Assigned_Location"].head(8).tolist()

    closed_apls = [
        apl_id for apl_id in apl_locations.keys()
        if apl_id not in current_open
    ]

    for loc_to_close in close_candidates:

        for apl_id in closed_apls:

            test_open = current_open.copy()
            test_open.remove(loc_to_close)
            test_open.add(apl_id)

            test_result = evaluate_network(test_open)

            improvement = current["total_cost"] - test_result["total_cost"]

            if test_result["feasible"] and improvement > 0:

                if best_move is None or improvement > best_move["improvement"]:
                    best_move = {
                        "operator": "CLOSE+OPEN",
                        "location": f"{loc_to_close} -> {apl_id}",
                        "result": test_result,
                        "improvement": improvement
                    }

    # =================================================
    # Accept best move
    # =================================================

    if best_move is None:
        print(f"\nNo improving move found at iteration {iteration}.")
        break

    old_cost = current["total_cost"]
    current = best_move["result"]

    search_log.append({
        "Iteration": iteration,
        "Operator": best_move["operator"],
        "Move": best_move["location"],
        "Old_Cost": old_cost,
        "New_Cost": current["total_cost"],
        "Improvement": best_move["improvement"],
        "Open_Locations": len(current["open_locations"]),
        "Avg_Distance": current["avg_distance"],
        "Max_Distance": current["max_distance"]
    })

    print(f"\nIteration {iteration}")
    print(f"Operator: {best_move['operator']}")
    print(f"Move: {best_move['location']}")
    print(f"Improvement: €{best_move['improvement']:,.0f}")
    print(f"New cost: €{current['total_cost']:,.0f}")
    print(f"Open locations: {len(current['open_locations'])}")
    print(f"Avg distance: {current['avg_distance']:.2f} km")
    print(f"Max distance: {current['max_distance']:.2f} km")


# -----------------------------------------------------
# Final results
# -----------------------------------------------------

final = current
search_log_df = pd.DataFrame(search_log)
final_solution = final["solution"]
final_summary = location_summary(final)

print("\nFINAL FULL LOCAL SEARCH SOLUTION")
print("=" * 60)
print(f"Baseline total cost: €{best['total_cost']:,.0f}")
print(f"Final total cost: €{final['total_cost']:,.0f}")
print(f"Cost reduction: €{best['total_cost'] - final['total_cost']:,.0f}")
print(f"Cost reduction %: {(best['total_cost'] - final['total_cost']) / best['total_cost'] * 100:.2f}%")

print(f"\nBaseline open locations: {len(best['open_locations'])}")
print(f"Final open locations: {len(final['open_locations'])}")

print(f"\nBaseline avg distance: {best['avg_distance']:.2f} km")
print(f"Final avg distance: {final['avg_distance']:.2f} km")

print(f"\nBaseline max distance: {best['max_distance']:.2f} km")
print(f"Final max distance: {final['max_distance']:.2f} km")

print("\nSearch log:")
display(search_log_df)

print("\nFinal location summary:")
display(final_summary)

print("\nFinal assignment table:")
display(final_solution.head(20))

Created 20 candidate APLs.
Precomputing distances to SPs and APLs...
Distance precomputation completed.
INITIAL NETWORK
Open locations: 35
Total cost: €2,543,555
Fixed cost: €1,750,000
Variable cost: €793,555
Average distance: 1.44 km
Maximum distance: 5.94 km

Least efficient baseline locations:


,Assigned_Location,Population,Demand,Pickups,Deliveries,Avg_Distance,Max_Distance,Variable_Cost,Squares,Fixed_Cost,Type,Total_Location_Cost,Cost_per_Parcel
32,7422,1105.0,11099.691336,8435.263167,2664.428169,1.305361,1.669106,5738.592693,2,50000,SP,55738.592693,5.021634
9,1258,1060.0,10647.667707,8299.656078,2348.011629,1.112221,1.917994,2624.871848,5,50000,SP,52624.871848,4.942385
7,1176,1565.0,15720.377322,12365.357497,3355.019825,1.001146,1.647670,3975.844868,2,50000,SP,53975.844868,3.433496
29,6767,1645.0,16523.974885,12706.886462,3817.088423,0.800198,1.191689,5839.437422,3,50000,SP,55839.437422,3.379298
30,6908,1620.0,16272.850646,12882.673428,3390.177218,0.688858,1.098680,4041.554298,3,50000,SP,54041.554298,3.320964
20,4354,1690.0,16975.998514,13580.798811,3395.199703,0.600680,0.910625,4398.543173,2,50000,SP,54398.543173,3.204438
19,3921,1705.0,17126.673057,13686.270991,3440.402066,0.792490,1.795080,3713.634622,7,50000,SP,53713.634622,3.136256
28,5820,14285.0,143492.389805,70849.681372,72642.708434,3.094782,5.943292,377643.696694,33,50000,SP,427643.696694,2.980253
1,417,1800.0,18080.945163,14464.756130,3616.189033,0.395226,0.615252,3201.444481,3,50000,SP,53201.444481,2.942404
5,931,1940.0,19487.240898,15589.792718,3897.448180,0.619811,0.904076,4677.089226,2,50000,SP,54677.089226,2.805789



Iteration 1
Operator: CLOSE+OPEN
Move: 1258 -> APL_E1805N3220
Improvement: €289,464
New cost: €2,254,090
Open locations: 35
Avg distance: 1.23 km
Max distance: 4.08 km

Iteration 2
Operator: CLOSE
Move: 6908
Improvement: €44,761
New cost: €2,209,329
Open locations: 34
Avg distance: 1.23 km
Max distance: 4.08 km

Iteration 3
Operator: CLOSE
Move: 3513
Improvement: €47,929
New cost: €2,161,401
Open locations: 33
Avg distance: 1.24 km
Max distance: 4.08 km

Iteration 4
Operator: CLOSE
Move: 4356
Improvement: €44,232
New cost: €2,117,169
Open locations: 32
Avg distance: 1.25 km
Max distance: 4.08 km

Iteration 5
Operator: CLOSE
Move: 7422
Improvement: €43,809
New cost: €2,073,361
Open locations: 31
Avg distance: 1.25 km
Max distance: 4.08 km

Iteration 6
Operator: CLOSE
Move: 1176
Improvement: €43,405
New cost: €2,029,956
Open locations: 30
Avg distance: 1.26 km
Max distance: 4.08 km

Iteration 7
Operator: CLOSE
Move: 1934
Improvement: €46,051
New cost: €1,983,904
Open locations: 29
Avg d

,Iteration,Operator,Move,Old_Cost,New_Cost,Improvement,Open_Locations,Avg_Distance,Max_Distance
0,1,CLOSE+OPEN,1258 -> APL_E1805N3220,2.543555e+06,2.254090e+06,289464.450280,35,1.229344,4.077157
1,2,CLOSE,6908,2.254090e+06,2.209329e+06,44760.652015,34,1.234107,4.077157
2,3,CLOSE,3513,2.209329e+06,2.161401e+06,47928.643964,33,1.236785,4.077157
3,4,CLOSE,4356,2.161401e+06,2.117169e+06,44231.512906,32,1.247065,4.077157
4,5,CLOSE,7422,2.117169e+06,2.073361e+06,43808.737285,31,1.252503,4.077157
5,6,CLOSE,1176,2.073361e+06,2.029956e+06,43404.917691,30,1.256160,4.077157
6,7,CLOSE,1934,2.029956e+06,1.983904e+06,46051.272009,29,1.258541,4.077157
7,8,CLOSE,1026,1.983904e+06,1.941533e+06,42371.593064,28,1.278415,4.077157
8,9,CLOSE,417,1.941533e+06,1.900160e+06,41372.348733,27,1.287892,4.077157
9,10,CLOSE,4987,1.900160e+06,1.855734e+06,44425.938713,26,1.292854,4.077157



Final location summary:


,Assigned_Location,Population,Demand,Pickups,Deliveries,Avg_Distance,Max_Distance,Variable_Cost,Squares,Fixed_Cost,Type,Total_Location_Cost,Cost_per_Parcel
20,6767,1645.0,16523.974885,12706.886462,3817.088423,0.800198,1.191689,5839.437422,3,50000,SP,55839.437422,3.379298
13,4354,1690.0,16975.998514,13580.798811,3395.199703,0.600680,0.910625,4398.543173,2,50000,SP,54398.543173,3.204438
12,3921,1705.0,17126.673057,13686.270991,3440.402066,0.792490,1.795080,3713.634622,7,50000,SP,53713.634622,3.136256
0,284,2435.0,24459.500817,18369.738037,6089.762781,1.342353,3.219903,12332.071891,13,50000,SP,62332.071891,2.548379
10,3155,2755.0,27673.891069,20421.423064,7252.468004,1.172110,2.089440,11895.632763,10,50000,SP,61895.632763,2.236608
8,2096,2765.0,27774.340764,21149.683356,6624.657408,1.801229,4.077157,11791.912743,14,50000,SP,61791.912743,2.224784
1,657,3740.0,37568.186060,27930.037792,9638.148269,1.674279,3.465803,23630.229475,15,50000,SP,73630.229475,1.959909
4,1204,3025.0,30386.032843,24308.826274,6077.206569,0.402748,0.634057,5110.682784,4,50000,SP,55110.682784,1.813685
22,7859,4795.0,48165.628920,35805.293907,12360.335013,1.427205,3.720060,24569.654915,11,50000,SP,74569.654915,1.548192
7,2075,3770.0,37869.535146,30295.628117,7573.907029,0.622495,0.892392,8615.629449,4,50000,SP,58615.629449,1.547831



Final assignment table:


,Square,Population,Assigned_Location,Distance_km,Annual_Demand,Pickup_Probability,Pickups,Deliveries,Variable_Cost
0,E1725N3170,175.0,2075,0.892392,1757.869669,0.80,1406.295735,351.573934,611.242422
1,E1730N3160,435.0,1379,0.612982,4369.561748,0.80,3495.649398,873.912350,1153.103556
2,E1730N3165,1815.0,1379,0.631173,18231.619706,0.80,14585.295765,3646.323941,4910.721615
3,E1730N3170,1975.0,2075,0.518720,19838.814831,0.80,15871.051865,3967.762966,4674.345030
4,E1730N3175,140.0,2075,0.689311,1406.295735,0.80,1125.036588,281.259147,403.316099
5,E1730N3185,90.0,5706,1.211722,904.047258,0.65,587.630718,316.416540,633.876344
6,E1730N3190,1050.0,5706,1.185865,10547.218012,0.65,6855.691708,3691.526304,7252.049187
7,E1735N3155,630.0,1379,1.351380,6328.330807,0.65,4113.415025,2214.915782,4901.131560
8,E1735N3160,650.0,1379,0.782467,6529.230198,0.80,5223.384158,1305.846040,2055.010067
9,E1735N3165,1895.0,1379,0.155907,19035.217269,0.80,15228.173815,3807.043454,2413.134890


In [24]:
print(final_summary["Demand"].sum())
print(TOTAL_PARCELS)

1351852.0
1351852


In [25]:
print(final_solution["Population"].sum())
print(cbs_clean["Population"].sum())

134580.0
134580.0


In [26]:
print(final_solution["Assigned_Location"].nunique())

25
